# 01 — Build the dataset: inputs, outputs, and the on-disk cache

Define the **data contract** for the model and materialize it to a fast cache that
`02_train_model.ipynb` trains on. Written to be *read*: each step prints the shape/dtype
it produces, so the preprocessing is fully legible.

**Learning problem (v1).** For each calendar day **D** with ≥1 reported flood, predict
**which 50 km CONUS-land cells flood on D** from the **previous day's (D−1)** GOES
imagery. One sample = one `(D−1 imagery → D flood map)` pair.

**The data contract**

| tensor | shape | dtype | meaning |
|---|---|---|---|
| `x` GOES | `(T=6, 6, 1500, 2500)` | f16 | 6 daytime frames × 6 ABI bands on the 2 km grid |
| `t` lead | `(6,)` | f32 | per-frame lead time (hours before D 06:00 UTC) |
| `y` label | `(59, 95)` | uint8 | 0/1 flood per 50 km cell, scored on land only |

> **Bands** = `config.BANDS`. **Lead time** is appended as a 7th input channel in
> notebook 02 (so the model knows how far ahead each frame sits). **Location** is added
> in the model as a learned per-cell embedding (not here). **GLM lightning** is deferred.

Pipeline: build the **output grid** → rasterize **labels** → read & normalize **GOES**
→ assemble one **sample** → index all valid samples → **materialize the cache**.

## 0. Config & imports

All knobs live in the repo-root `config.py` (single source of truth). The lever that
matters most is `CELL_KM` (output cell size) — everything downstream derives from it.

In [ ]:
import json
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import pandas as pd

# all shared constants live in the repo-root config.py (single source of truth)
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from config import (BANDS, CACHE_DIR, CELL_KM, DATA_DIR, IMG_H, IMG_W, N_BAND,
                    SPLIT_FRACS, SPLIT_SEED, STATS_PATH, T_FRAMES,
                    UNIFIED_PARQUET, YEAR, build_grid_cells)

# inputs: a 6-frame GOES band sequence + a per-frame lead time;
# output: the next-day flood map on the CELL_KM grid
print(f"GOES seq : (T={T_FRAMES}, {N_BAND}, {IMG_H}, {IMG_W})  "
      f"= {T_FRAMES*N_BAND*IMG_H*IMG_W*4/1e6:.0f} MB f32  (+ {T_FRAMES} lead hrs)")
print(f"output   : {CELL_KM} km cells | bands {BANDS} | {YEAR}")

## 1. Output target — the 50 km cell grid

The label space is a fixed grid of square **50 km** cells over CONUS land, generated on
the fly by `config.build_grid_cells()` in an equal-area projection (EPSG:5070) and kept
where they intersect land. Nothing is stored — it's a pure function of `CELL_KM` + the
CONUS boundary, so notebook 02 rebuilds the identical grid. Every label `y` is a
`(GRID_R, GRID_C)` image; the loss is scored only on the `land_mask` cells.

> **Orientation:** row 0 = north, col 0 = west — matching the GOES imagery, so inputs and
> labels share one frame and plots render right-way-up.

In [ ]:
# generated fresh at CELL_KM over CONUS land (config.build_grid_cells; nothing stored)
cells, GRID_R, GRID_C, land_mask = build_grid_cells()

print(f"{CELL_KM} km grid: ({GRID_R}, {GRID_C})  <- every y map is this shape")
print(f"land cells: {int(land_mask.sum())} of {GRID_R*GRID_C}  <- loss computed here")

plt.figure(figsize=(10, 8))
plt.imshow(land_mask, cmap="Greys", interpolation="none")
plt.title(f"land mask  ({GRID_R}x{GRID_C}, {int(land_mask.sum())} cells)")
plt.xlabel("col (W->E)"); plt.ylabel("row (N->S)"); plt.tight_layout()

## 2. Labels — flood events → daily 0/1 cell maps

Rasterize flood events onto the grid: each event's footprint marks every 50 km cell it
**intersects** with a 1, stamped on the event's start day → `labels[day] → (GRID_R, GRID_C)`.

> **Label source matters.** `floods_unified.parquet` unions three layers — observed
> (groundsource news), predicted (NWS warnings), confirmed (NCEI storm events). This v1
> cache uses the **union** for a denser target (~2% positive). The cleaner single-source
> target is **storm events only** (confirmed, ~0.2% positive); change the filter below to
> switch. The choice materially changes the task and its metrics.

In [ ]:
u = gpd.read_parquet(UNIFIED_PARQUET)
ev = u[u["issue_date"].dt.year == YEAR].copy()         # ALL sources
ev["label_day"] = ev["issue_date"].dt.normalize()      # day resolution

# spatial join: which 50 km cell does each event footprint touch?
j = gpd.sjoin(cells, ev[["label_day", "geometry"]], predicate="intersects")

labels = {}
for day, g in j.groupby("label_day"):
    a = np.zeros((GRID_R, GRID_C), dtype=np.float32)
    a[g["R"], g["C"]] = 1.0
    labels[day.date()] = a

pos_per_day = np.array([a[land_mask].sum() for a in labels.values()])
print(f"{len(ev):,} flood events (all layers) on {len(labels)} days in {YEAR}")
print("by source:", ev["source"].value_counts().to_dict())
print(f"flooded land cells/day: median {np.median(pos_per_day):.0f}, "
      f"max {pos_per_day.max():.0f} "
      f"({np.median(pos_per_day)/land_mask.sum():.1%} of land)")

In [ ]:
# look at the busiest flood day's label map
busy_day = max(labels, key=lambda d: labels[d].sum())
ymap = labels[busy_day]
shown = np.where(land_mask, ymap, np.nan)          # grey-out ocean for clarity

plt.figure(figsize=(10, 8))
plt.imshow(np.where(land_mask, 0.15, np.nan), cmap="Greys", vmin=0, vmax=1)
plt.imshow(shown, cmap="Reds", vmin=0, vmax=1, interpolation="none")
plt.title(f"y for {busy_day}  ->  shape {ymap.shape}, "
          f"{int(ymap.sum())} flooded cells")
plt.xlabel("col"); plt.ylabel("row"); plt.tight_layout()

## 3. Input — GOES ABI bands

GOES files are one NetCDF per scan (`MCMIPC`, all 16 ABI bands inside). Per day we keep
the **6 daytime frames** (16–21 UTC) and the **bands in `config.BANDS`**. Below: the
filename→time helpers, then every selected band's raw values for one frame.

In [ ]:
def _scan_token(p):
    "Return the s-token (s{YYYYDDDHHMM...}) from a GOES filename."
    for part in p.name.split("_"):
        if part.startswith("s") and part[1:].isdigit():
            return part
    return p.name


def _scan_dt(p):
    "Scan-start datetime parsed from the filename token (UTC)."
    return datetime.strptime(_scan_token(p)[1:12], "%Y%j%H%M")


def goes_files(d):
    "Sorted list of the day's GOES NetCDFs (GOES16 or GOES19 auto-globbed)."
    pat = f"*/{d.year}/{d.month:02d}/{d.day:02d}/*.nc"
    return sorted(DATA_DIR.glob(pat), key=_scan_token)


# pick one example day that has all 6 frames the day before (a real sample)
example_label_day = sorted(d for d in labels
                           if len(goes_files(d - timedelta(days=1))) == T_FRAMES)[5]
example_in_day = example_label_day - timedelta(days=1)
files = goes_files(example_in_day)
print(f"input day {example_in_day}  ->  label day {example_label_day}")
print(f"{len(files)} frames; scan times (UTC):",
      [f"{_scan_dt(f):%H:%M}" for f in files])

In [ ]:
# read ALL selected bands from ONE frame, to see each band's raw values
mid = files[3]                                   # a midday frame
# band -> (short name, units, colormap); 1/2/3/6 are reflectance, 13/16 are IR temp
BAND_INFO = {
    1:  ("blue 0.47um",       "reflectance",      "Blues"),     # blue band -> blue
    2:  ("red 0.64um",        "reflectance",      "Reds"),      # red band  -> red
    3:  ("veggie NIR 0.86um", "reflectance",      "Greens"),    # vegetation -> green
    6:  ("cloud size 2.2um",  "reflectance",      "Purples"),   # cloud/snow -> purple
    7:  ("SW window 3.9um",   "brightness T (K)", "inferno"),   # shortwave window
    10: ("low WV 7.3um",      "brightness T (K)", "BuPu"),      # low water vapour
    13: ("clean IR 10.3um",   "brightness T (K)", "RdYlBu_r"),  # temp: cold=blue, warm=red
    16: ("CO2 13.3um",        "brightness T (K)", "Spectral_r"),# temp (distinct palette)
}

with netCDF4.Dataset(mid) as nc:
    raws = {b: np.ma.filled(nc[f"CMI_C{b:02d}"][:], np.nan) for b in BANDS}

ncols = 3
nrows = int(np.ceil(len(BANDS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 3.1 * nrows))
for ax, b in zip(axes.flat, BANDS):
    name, units, cmap = BAND_INFO[b]
    a = raws[b]
    im = ax.imshow(a[::4, ::4], cmap=cmap)       # subsample just for display
    ax.set_title(f"band {b} — {name}", fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.85, label=units)
    ax.set_xticks([]); ax.set_yticks([])
    print(f"band {b:>2} ({name:<16}): min {np.nanmin(a):7.2f}  max {np.nanmax(a):7.2f}"
          f"  [{units}]")
for ax in axes.flat[len(BANDS):]:                # hide any unused panels
    ax.axis("off")
fig.suptitle(f"all {len(BANDS)} selected GOES bands  {example_in_day} "
             f"{_scan_dt(mid):%H:%M}Z", fontsize=11)
plt.tight_layout()

### Per-band normalization

Bands live on very different scales (visible reflectance 0–1, IR brightness temps
~180–320 K). Standardize each band to ~zero-mean/unit-variance using per-band statistics
from a handful of training frames (cached to JSON; recomputed automatically when `BANDS`
changes). The model always sees standardized inputs.

In [ ]:
if STATS_PATH.exists():
    stats = json.loads(STATS_PATH.read_text())
    print(f"loaded cached band stats from {STATS_PATH.name}")
else:
    # sample a few midday frames across the year and measure mean/std per band
    cand = sorted(d - timedelta(days=1) for d in labels
                  if len(goes_files(d - timedelta(days=1))) == T_FRAMES)
    rng = np.random.default_rng(0)
    pick = rng.choice(len(cand), size=8, replace=False)
    acc = {b: [] for b in BANDS}
    for i in pick:
        with netCDF4.Dataset(goes_files(cand[i])[3]) as nc:
            for b in BANDS:
                a = np.ma.filled(nc[f"CMI_C{b:02d}"][:], np.nan)
                acc[b].append(a[::4, ::4])          # subsample is plenty
    stats = {str(b): {"mean": float(np.nanmean(np.stack(acc[b]))),
                      "std": float(np.nanstd(np.stack(acc[b])))} for b in BANDS}
    STATS_PATH.write_text(json.dumps(stats, indent=2))
    print(f"computed and cached band stats -> {STATS_PATH}")

BAND_MEAN = np.array([stats[str(b)]["mean"] for b in BANDS], dtype=np.float32)
BAND_STD = np.array([stats[str(b)]["std"] for b in BANDS], dtype=np.float32)
for b, m, s in zip(BANDS, BAND_MEAN, BAND_STD):
    print(f"  band {b:>2}: mean {m:8.3f}  std {s:7.3f}")

## 4. Assemble one full sample

`load_sample(in_day, label_day)` reads the 6 frames, normalizes each band, computes each
frame's lead time, and returns the `(x, t, y)` triple of the data contract. Off-Earth
pixels are zeroed after normalization.

In [ ]:
def load_sample(in_day, label_day):
    files = goes_files(in_day)[:T_FRAMES]
    bands = np.zeros((T_FRAMES, N_BAND, IMG_H, IMG_W), dtype=np.float32)
    for t, f in enumerate(files):
        with netCDF4.Dataset(f) as nc:
            for c, b in enumerate(BANDS):
                a = np.ma.filled(nc[f"CMI_C{b:02d}"][:], np.nan)
                bands[t, c] = (a - BAND_MEAN[c]) / BAND_STD[c]   # normalize band
    np.nan_to_num(bands, copy=False)
    # per-frame lead time: hours before day D's CST start (= D 06:00 UTC)
    d06 = datetime(label_day.year, label_day.month, label_day.day, 6)
    frame_lead = np.array([(d06 - _scan_dt(f)).total_seconds() / 3600
                           for f in files], np.float32)
    return bands, frame_lead, labels[label_day]


t0 = time.perf_counter()
bands, frame_lead, y = load_sample(example_in_day, example_label_day)
dt = time.perf_counter() - t0
print(f"GOES seq   {bands.shape}  {bands.dtype}  = {bands.nbytes/1e6:.0f} MB")
print(f"frame_lead {frame_lead.shape}  = {frame_lead.round(1)}")
print(f"y          {y.shape}  = {int(y.sum())} flooded cells")
print(f"loaded one sample in {dt:.1f}s  (NetCDF decode)")

In [ ]:
# visualize: clean-IR across the 6 frames + the label
fig, axes = plt.subplots(1, T_FRAMES + 1, figsize=(2.0 * (T_FRAMES + 1), 2.4))
for t in range(T_FRAMES):
    axes[t].imshow(bands[t, 4, ::6, ::6], cmap="gray_r")   # channel 4 = band 13
    axes[t].set_title(f"-{frame_lead[t]:.0f}h", fontsize=8)
    axes[t].axis("off")
axes[-1].imshow(np.where(land_mask, 0.15, np.nan), cmap="Greys", vmin=0, vmax=1)
axes[-1].imshow(np.where(land_mask, y, np.nan), cmap="Reds", vmin=0, vmax=1)
axes[-1].set_title(f"y: {example_label_day}", fontsize=8)
axes[-1].axis("off")
fig.suptitle("one sample:  band-13 sequence (lead hrs, D-1)   ->   floods (D)",
             fontsize=10)
plt.tight_layout()

## 5. Sample index + train / val / test split

A sample is valid only if **all 6 GOES frames** exist on day D−1. We index every valid
`(D−1 → D)` pair and split it; each sample is an independent day-to-next-day prediction.

> **Split caveat.** The split written here is random (reproducible). Adjacent days are
> weather-correlated, so a random split is slightly optimistic — notebook 02 uses a
> **temporal** split (train earlier days, test later) for the honest estimate.

In [ ]:
SPLIT_SEED = 0
SPLIT_FRACS = (0.70, 0.20, 0.10)            # train / val / test

rows = []
for d_label in sorted(labels):
    d_in = d_label - timedelta(days=1)
    if len(goes_files(d_in)) == T_FRAMES:
        rows.append({"in_day": pd.Timestamp(d_in),
                     "label_day": pd.Timestamp(d_label),
                     "n_pos": int(labels[d_label].sum())})

index = pd.DataFrame(rows)

# reproducible random 70/20/10 assignment
rng = np.random.default_rng(SPLIT_SEED)
perm = rng.permutation(len(index))
n_tr = int(SPLIT_FRACS[0] * len(index))
n_val = int(SPLIT_FRACS[1] * len(index))
split = np.array(["test"] * len(index), dtype=object)
split[perm[:n_tr]] = "train"
split[perm[n_tr:n_tr + n_val]] = "val"
index["split"] = split

print(index["split"].value_counts().reindex(["train", "val", "test"]).to_string())
print(f"\ntotal samples: {len(index)}")

# class balance over the train split -> pos_weight for notebook 02's loss
tr = index[index["split"] == "train"]
tr_y = np.stack([labels[d.date()] for d in tr["label_day"]])[:, land_mask]
pos_rate = float(tr_y.mean())
print(f"train positive rate: {pos_rate:.3%}  "
      f"-> suggested pos_weight ~= {min((1-pos_rate)/pos_rate, 100):.0f}")
index.head()

## 6. Materialize the cache

Reading 6 NetCDFs per sample costs seconds *per epoch*. Pay it once: precompute every
sample and write `x` (float16, ~half size), the lead times, and the label as separate
`.npy` files, parallelized across CPU cores. Notebook 02 then memory-maps these and
trains ~10× faster. Set `BUILD_CACHE=True` to (re)build all samples — **rebuild after any
change to `BANDS` or the label source.**

In [ ]:
from concurrent.futures import ProcessPoolExecutor

CACHE_DIR.mkdir(parents=True, exist_ok=True)
index.to_parquet(CACHE_DIR / "manifest.parquet")        # splits for notebook 02
print(f"wrote manifest ({len(index)} samples) -> {CACHE_DIR}")
# (the grid + land mask are regenerated from config.build_grid_cells(), not stored)

BUILD_CACHE = True                  # <- flip to True to build ALL samples
N_WORKERS = 24                       # CPU processes (box has 64 cores)


def _write_one(args):
    in_day, label_day = args
    stem = CACHE_DIR / f"{label_day:%Y%m%d}"
    if Path(f"{stem}_x.npy").exists():
        return 0
    bands, frame_lead, y = load_sample(in_day, label_day)
    np.save(f"{stem}_x.npy", bands.astype(np.float16))          # GOES sequence
    np.save(f"{stem}_t.npy", frame_lead)                        # frame lead-hours
    np.save(f"{stem}_y.npy", y.astype(np.uint8))
    return bands.nbytes // 2


todo = list(zip(index["in_day"].dt.date, index["label_day"].dt.date))
if not BUILD_CACHE:
    todo = todo[:2]
    print(f"smoke test: writing {len(todo)} samples "
          "(set BUILD_CACHE=True for all)")
else:
    gb = len(todo) * bands.nbytes / 2 / 1e9
    print(f"building ALL {len(todo)} samples (~{gb:.0f} GB) "
          f"on {N_WORKERS} workers ...")

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    written = list(ex.map(_write_one, todo))
print(f"done: {sum(b > 0 for b in written)} new files in "
      f"{(time.perf_counter()-t0)/60:.1f} min  -> {CACHE_DIR}")

---
### Data contract recap

The cache under `cache/floodnet_{YEAR}/` holds, per sample day:
- `{date}_x.npy` `(6, 6, 1500, 2500)` f16 — 6 frames × `BANDS`
- `{date}_t.npy` `(6,)` f32 — per-frame lead times
- `{date}_y.npy` `(59, 95)` uint8 — flood map on the 50 km grid

plus `manifest.parquet` (the split). The grid + land mask are regenerated from
`config.build_grid_cells()`, never stored.

**Next:** `02_train_model.ipynb` appends the lead-time + location channels and trains the
ConvLSTM / CNN-LSTM / ResNet models.